In [3]:
import pandas as pd
from PIL import Image
import torch
from transformers import BlipProcessor, BlipModel
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score, f1_score, classification_report, confusion_matrix
import numpy as np
from tqdm import tqdm

# Load CSV
csv_path = "Complete_Img.csv"
df = pd.read_csv(csv_path)
df['label'] = df['label'].map({'Non_hope_speech': 0, 'Hope_speech': 1})

# Split dataset
# Ensure stratified split based on labels
# Using 30% for validation and test sets, 70% for training
#we use stratify = temp_df['label'] to ensure that the split maintains the same proportion of classes in each set.
train_df, temp_df = train_test_split(df, test_size=0.3, random_state=50, shuffle=True,stratify=df['label'])
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=50, shuffle=True, stratify=temp_df['label'])
print(f"Train size: {len(train_df)}, Val size: {len(val_df)}, Test size: {len(test_df)}")

# Load BLIP model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
model = BlipModel.from_pretrained("Salesforce/blip-image-captioning-base").to(device)
model.eval()

# Function to get concatenated image+text embeddings as 1D numpy array
# This function processes an image and text, extracts embeddings, normalizes them, and concatenates them.
def get_concat_embeddings(image, text):# Process image and text to get embeddings
    # Convert image to RGB if not already
    inputs = processor(image, text, return_tensors="pt", padding=True).to(device)# Convert inputs to tensors and move to device
    # Forward pass through the model to get embeddings
    with torch.no_grad():# Disable gradient calculation for inference
        # outputs is a tuple containing image_embeds and text_embeds
        outputs = model(**inputs)
        # normalize separately
        image_embeds = outputs.image_embeds / outputs.image_embeds.norm(p=2, dim=-1, keepdim=True)
        text_embeds = outputs.text_embeds / outputs.text_embeds.norm(p=2, dim=-1, keepdim=True)
    concat_embeds = torch.cat([image_embeds, text_embeds], dim=-1)
    return concat_embeds.squeeze(0).cpu().numpy()# Convert to numpy array and squeeze to 1D

# Extract embeddings and labels for a dataframe
# This function iterates through each row of the dataframe, processes the image and text, and collects the embeddings and labels.
def extract_features_labels(df):
    features = []
    labels = []
    for idx, row in tqdm(df.iterrows(), total=len(df)):# tqdm provides a progress bar for the loop
        # Try-except block to handle any issues with image loading or processing
        try:
            img = Image.open(row['image']).convert('RGB')# Convert image to RGB
            # Get concatenated embeddings
            emb = get_concat_embeddings(img, row['text'])# Extract embeddings
            # Normalize the embeddings
            features.append(emb)# Normalize the embeddings
            # Append the label
            labels.append(row['label'])
        except Exception as e:# Handle exceptions for image loading or processing
            # Print the error and skip this row
            print(f"Skipping idx {idx} due to {e}")
    return np.array(features), np.array(labels)# Return the features and labels as numpy arrays

# Extract features and labels
# This part processes the training, validation, and test dataframes to extract the embeddings and labels.
X_train, y_train = extract_features_labels(train_df)
X_val, y_val = extract_features_labels(val_df)
X_test, y_test = extract_features_labels(test_df)

#print("Training Logistic Regression classifier...")

# Train logistic regression
# Using a logistic regression model with class weights to handle class imbalance.

# clf = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=50)
# clf.fit(X_train, y_train)

# # Evaluate on validation set
# # This part evaluates the trained model on the validation set and prints the classification report.
# val_preds = clf.predict(X_val)
# print("\nValidation set classification report:")
# print(classification_report(y_val, val_preds, target_names=["Non", "Hope"], digits=4))

# # Evaluate on test set
# test_preds = clf.predict(X_test)
# print("\nTest set classification report:")
# print(classification_report(y_test, test_preds, target_names=["Non", "Hope"], digits=4))
# print("\nConfusion matrix on test set:")
# print(confusion_matrix(y_test, test_preds))



##------------------------Train and Evaulation (RF)--------------------------------##
#Random Forest model for classification
# clf = RandomForestClassifier(n_estimators=250).fit(X_train, y_train)

# y_pred = clf.predict(X_test)

# f1 = f1_score(y_test, y_pred, average='weighted')
# precision = precision_score(y_test, y_pred, average='weighted')
# recall = recall_score(y_test, y_pred, average='weighted')

# print(f"Precision: {precision:.4f}")
# print(f"Recall:    {recall:.4f}")
# print(f"F1 Score:  {f1:.4f}")
# print(classification_report(y_test, y_pred, digits=4))

##------------------------Train and Evaulation (XGB)--------------------------------##
#XGBoost model for classification
#need to translate hope and non hope into 1,0 



clf = XGBClassifier(n_estimators=250).fit(X_train, y_train)

y_predE = clf.predict(X_test)

f1 = f1_score(y_test, y_predE, average='weighted')
precision = precision_score(y_test, y_predE, average='weighted')
recall = recall_score(y_test, y_predE, average='weighted')

print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 Score:  {f1:.4f}")
print(classification_report(y_test, y_predE, digits=4))



Train size: 2717, Val size: 582, Test size: 583


`BlipModel` is going to be deprecated in future release, please use `BlipForConditionalGeneration`, `BlipForQuestionAnswering` or `BlipForImageTextRetrieval` depending on your usecase.
Some weights of BlipModel were not initialized from the model checkpoint at Salesforce/blip-image-captioning-base and are newly initialized: ['logit_scale', 'text_model.embeddings.LayerNorm.bias', 'text_model.embeddings.LayerNorm.weight', 'text_model.embeddings.position_embeddings.weight', 'text_model.embeddings.word_embeddings.weight', 'text_model.encoder.layer.0.attention.output.LayerNorm.bias', 'text_model.encoder.layer.0.attention.output.LayerNorm.weight', 'text_model.encoder.layer.0.attention.output.dense.bias', 'text_model.encoder.layer.0.attention.output.dense.weight', 'text_model.encoder.layer.0.attention.self.key.bias', 'text_model.encoder.layer.0.attention.self.key.weight', 'text_model.encoder.layer.0.attention.self.query.bias', 'text_model.encoder.layer.0.attention.self.query.weight', 'text_mo

Skipping idx 859 due to cannot identify image file 'images/IHS_0524.jpg'


  2%|▏         | 43/2717 [00:01<01:44, 25.53it/s]

Skipping idx 48 due to cannot identify image file 'images/IHS_0123.jpg'


...

Skipping idx 494 due to cannot identify image file 'images/IHS_0446.jpg'
Skipping idx 2801 due to cannot identify image file 'images/IHS_0360.jpg'


 91%|█████████▏| 532/583 [00:22<00:01, 26.35it/s]

Skipping idx 3371 due to cannot identify image file 'images/IHS_0070.jpg'


100%|██████████| 583/583 [00:24<00:00, 23.73it/s]


Precision: 0.8939
Recall:    0.8929
F1 Score:  0.8929
              precision    recall  f1-score   support

           0     0.9137    0.8699    0.8912       292
           1     0.8738    0.9164    0.8946       287

    accuracy                         0.8929       579
   macro avg     0.8937    0.8931    0.8929       579
weighted avg     0.8939    0.8929    0.8929       579

